# FINN/QONNX Flow Graph Debug Notebook

这个 notebook 的目的不是只把 PyTorch 模型导出成 ONNX，而是**逐步验证每个 graph-cleanup / graph-normalization 函数是否真的有帮助**。

每跑完一个阶段都看三类证据：

1. **Graph statistics**：节点数量、op histogram、MultiThreshold 数量、Add/Mul/Pad/Transpose/MatMul 等是否变化。
2. **Shape health**：zero-dim tensor 是否减少或消失。
3. **Functional equivalence**：PyTorch vs QONNX、QONNX vs FINN-ONNX 的 max/mean diff 和 argmax match 是否正常。

建议先按顺序全部运行一遍；如果某一步报错，就不要急着继续，先看该步骤前后的 summary。

## 0. 使用方式

默认假设这个 notebook 放在：

```text
finn/notebooks/onnx2bit/flow_graph_debug_notebook.ipynb
```

并且你的 TCN 项目在：

```text
finn/notebooks/pytorch-tcn
```

如果你的目录不同，只需要改下面 `PROJECT_DIR` 和 `CHECKPOINT_PATH`。

In [1]:

# =========================================================
# 0. Imports and path setup
# =========================================================

import sys
import os
from pathlib import Path
from collections import Counter, OrderedDict
import json
import time

import numpy as np
import torch
import onnx

from brevitas.export import export_qonnx

from qonnx.util.cleanup import cleanup
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import GiveUniqueNodeNames, GiveReadableTensorNames
from qonnx.transformation.infer_shapes import InferShapes

from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.streamline.absorb import (
    AbsorbAddIntoMultiThreshold,
    AbsorbMulIntoMultiThreshold,
    AbsorbSignBiasIntoMultiThreshold,
    AbsorbMulAddIntoMultiThreshold,
    DuplicateScalarMulAfterFork,
)

try:
    import pandas as pd
    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False

# In a notebook, __file__ does not exist. Use cwd.
THIS_DIR = Path.cwd().resolve()

# If this notebook is placed under finn/notebooks/onnx2bit, parent is finn/notebooks.
if THIS_DIR.name == "onnx2bit":
    NOTEBOOKS_DIR = THIS_DIR.parent
else:
    # fallback: assume current folder is finn/notebooks or run manually
    NOTEBOOKS_DIR = THIS_DIR

PROJECT_DIR = NOTEBOOKS_DIR / "pytorch-tcn"

sys.path.insert(0, str(PROJECT_DIR))

print("THIS_DIR      =", THIS_DIR)
print("NOTEBOOKS_DIR =", NOTEBOOKS_DIR)
print("PROJECT_DIR   =", PROJECT_DIR)
print("HAS_PANDAS    =", HAS_PANDAS)

from new_version.newnet import ECG5000FullQuantTCN

THIS_DIR      = /home/slowman/Desktop/project/Thesis/finn/notebooks/onnx2bit
NOTEBOOKS_DIR = /home/slowman/Desktop/project/Thesis/finn/notebooks
PROJECT_DIR   = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn
HAS_PANDAS    = True


In [2]:

# =========================================================
# 1. Fixed config
# =========================================================

CHECKPOINT_PATH = PROJECT_DIR / "new_version" / "best_qtcn_model_W4A4_finetune.pth"

EXPORT_DIR = PROJECT_DIR / "onnx_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RAW_QONNX_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_qonnx.onnx"
CLEAN_QONNX_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_qonnx_clean.onnx"
CLEAN_QONNX_SHAPED_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_qonnx_clean_shaped.onnx"
CLEAN_QONNX_SHAPED_FIXED_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx"
FINN_ONNX_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_finn.onnx"
FINN_ONNX_STREAMLINED_PATH = EXPORT_DIR / "ecg5000_pointwise_qtcn_finn_streamlined.onnx"

BATCH_SIZE = 1
IN_CH = 1
SEQ_LEN = 140
NUM_CLASSES = 5
DEVICE = "cpu"

# Set True when you want to regenerate all intermediate files.
FORCE_REEXPORT = True

print("CHECKPOINT_PATH =", CHECKPOINT_PATH)
print("EXPORT_DIR      =", EXPORT_DIR)
print("FORCE_REEXPORT  =", FORCE_REEXPORT)

CHECKPOINT_PATH = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/new_version/best_qtcn_model_W4A4_finetune.pth
EXPORT_DIR      = /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports
FORCE_REEXPORT  = True


## 1. 基础输出转换函数

这些函数只负责把不同 layout 的 logits 统一成 `[N, C]`，方便做 PyTorch/QONNX/FINN 之间的数值比较。

如果你最终使用 B-scheme temporal classifier，理想输出应该是：

```text
[N, 5, 1, 1] -> [N, 5]
```

In [3]:

# =========================================================
# 2. Logits and ONNX output helpers
# =========================================================

def qt_value(x):
    return x.value if hasattr(x, "value") else x


def logits_to_2d_torch(x):
    """
    Convert PyTorch model output to [N, C] for accuracy comparison.

    Expected B-scheme output:
        [N, 5, 1, 1] -> [N, 5]
    """
    x = qt_value(x)

    if x.dim() == 2:
        return x

    if x.dim() == 4:
        if x.shape[1] == NUM_CLASSES and x.shape[2] == 1 and x.shape[3] == 1:
            return x[:, :, 0, 0]

        if x.shape[1] == NUM_CLASSES and x.shape[3] == 1:
            print("[warn] PyTorch model still outputs all time steps [N, C, L, 1]. Using last time step.")
            return x[:, :, -1, 0]

        if x.shape[-1] == NUM_CLASSES:
            print("[warn] PyTorch output looks NHWC-like. Using last time step.")
            return x[:, -1, 0, :]

    raise ValueError(f"Expected logits convertible to [N, C], got {tuple(x.shape)}")


def logits_to_2d_np(x):
    """
    Convert ONNX output to [N, C] for comparison.
    """
    x = np.asarray(x)

    if x.ndim == 2:
        return x

    if x.ndim == 4:
        if x.shape[1] == NUM_CLASSES and x.shape[2] == 1 and x.shape[3] == 1:
            return x[:, :, 0, 0]

        if x.shape[1] == NUM_CLASSES and x.shape[3] == 1:
            print("[warn] ONNX model still outputs all time steps [N, C, L, 1]. Using last time step.")
            return x[:, :, -1, 0]

        if x.shape[-1] == NUM_CLASSES and x.shape[2] == 1:
            print("[warn] ONNX output looks [N, L, 1, C]. Using last time step.")
            return x[:, -1, 0, :]

        if x.shape[-1] == NUM_CLASSES and x.shape[1] == 1:
            print("[warn] ONNX output looks [N, 1, L, C]. Using last time step.")
            return x[:, 0, -1, :]

    raise ValueError(f"Expected logits convertible to [N, C], got {x.shape}")


def pick_logits_output_from_onnx_outputs(output_dict, target_num_classes=NUM_CLASSES, tag=""):
    """
    Pick the output that looks like classification logits.
    """
    print(f"\n=== Available ONNX outputs: {tag} ===")
    for name, val in output_dict.items():
        arr = np.asarray(val)
        print(f"{name}: shape={arr.shape}")

    candidates = []

    for name, val in output_dict.items():
        arr = np.asarray(val)

        if arr.ndim == 2 and arr.shape[1] == target_num_classes:
            candidates.append((name, arr))
            continue

        if arr.ndim == 4:
            if arr.shape[1] == target_num_classes and arr.shape[2] == 1 and arr.shape[3] == 1:
                candidates.append((name, arr))
                continue

            if arr.shape[1] == target_num_classes and arr.shape[3] == 1:
                candidates.append((name, arr))
                continue

            if arr.shape[-1] == target_num_classes and arr.shape[2] == 1:
                candidates.append((name, arr))
                continue

            if arr.shape[-1] == target_num_classes and arr.shape[1] == 1:
                candidates.append((name, arr))
                continue

    if len(candidates) == 0:
        raise RuntimeError(
            f"Could not find logits-like output with class dim = {target_num_classes}."
        )

    chosen_name, chosen_val = candidates[0]
    converted = logits_to_2d_np(chosen_val)

    print(f"Chosen logits output: {chosen_name}, raw shape={chosen_val.shape}")
    print(f"Chosen logits converted shape={converted.shape}")

    return chosen_name, chosen_val

## 2. Graph 统计函数

这部分是 notebook 的核心之一。

每个阶段保存一个 ONNX 后，都跑 `print_graph_report(...)`。你主要看：

- `node_count` 是否减少；
- `zero_dim_tensor_count` 是否减少；
- `scalar_add_mul_count` 是否减少；
- `MultiThreshold` 数量和属性是否合理；
- `Pad/Transpose/Reshape/Mul/Add` 是否还很多。

这能回答：“某个函数是不是真的整理了 flow graph？”

In [4]:

# =========================================================
# 3. Graph statistics and inspection helpers
# =========================================================

QUANT_LIKE_OPS = [
    "Quant",
    "BinaryQuant",
    "Trunc",
    "IntQuant",
    "BipolarQuant",
    "FloatQuant",
    "MultiThreshold",
]

INTERESTING_OPS = [
    "Quant",
    "MultiThreshold",
    "Conv",
    "MatMul",
    "Im2Col",
    "Transpose",
    "Reshape",
    "Pad",
    "Mul",
    "Add",
    "Sub",
    "Div",
    "Relu",
    "MaxPool",
    "BatchNormalization",
]


def _normalize_shape(shape):
    if shape is None:
        return None
    out = []
    for d in shape:
        try:
            out.append(int(d))
        except Exception:
            out.append(str(d))
    return out


def _shape_has_zero(shape):
    if shape is None:
        return False
    for d in shape:
        try:
            if int(d) == 0:
                return True
        except Exception:
            pass
    return False


def collect_tensor_names(model):
    names = set()

    for vi in list(model.graph.input) + list(model.graph.output) + list(model.graph.value_info):
        names.add(vi.name)

    for init in model.graph.initializer:
        names.add(init.name)

    for node in model.graph.node:
        for x in node.input:
            if x != "":
                names.add(x)
        for x in node.output:
            if x != "":
                names.add(x)

    return sorted(names)


def get_initializer_shape(model, name):
    try:
        arr = model.get_initializer(name)
        if arr is None:
            return None
        return list(arr.shape)
    except Exception:
        return None


def is_scalar_initializer(model, name):
    shape = get_initializer_shape(model, name)
    if shape is None:
        return False
    return shape == [] or shape == [1] or np.prod(shape) == 1


def count_scalar_add_mul_nodes(model):
    count = 0
    names = []
    for node in model.graph.node:
        if node.op_type not in ["Add", "Mul"]:
            continue
        if any(is_scalar_initializer(model, x) for x in node.input):
            count += 1
            names.append(node.name if node.name else node.output[0])
    return count, names


def get_bad_shape_tensors(model):
    bad = []
    for name in collect_tensor_names(model):
        try:
            shape = _normalize_shape(model.get_tensor_shape(name))
        except Exception:
            continue
        if _shape_has_zero(shape):
            bad.append((name, shape))
    return bad


def graph_stats(onnx_path):
    model = ModelWrapper(str(onnx_path))
    ops = Counter([n.op_type for n in model.graph.node])

    bad_shapes = get_bad_shape_tensors(model)
    scalar_count, scalar_names = count_scalar_add_mul_nodes(model)

    quant_counts = {op: ops.get(op, 0) for op in QUANT_LIKE_OPS if ops.get(op, 0) > 0}
    interesting_counts = {op: ops.get(op, 0) for op in INTERESTING_OPS if ops.get(op, 0) > 0}

    stats = OrderedDict()
    stats["path"] = str(onnx_path)
    stats["exists"] = Path(onnx_path).is_file()
    stats["node_count"] = len(model.graph.node)
    stats["initializer_count"] = len(model.graph.initializer)
    stats["value_info_count"] = len(model.graph.value_info)
    stats["zero_dim_tensor_count"] = len(bad_shapes)
    stats["scalar_add_mul_count"] = scalar_count
    stats["multithreshold_count"] = ops.get("MultiThreshold", 0)
    stats["quant_like_counts"] = quant_counts
    stats["interesting_op_counts"] = interesting_counts
    stats["bad_shape_tensors"] = bad_shapes
    stats["scalar_add_mul_names"] = scalar_names
    stats["op_counts"] = dict(sorted(ops.items()))
    return stats


def print_model_io(onnx_path):
    model = ModelWrapper(str(onnx_path))

    print("Inputs:")
    for x in model.graph.input:
        try:
            shape = model.get_tensor_shape(x.name)
        except Exception:
            shape = None
        try:
            dt = model.get_tensor_datatype(x.name)
        except Exception:
            dt = None
        print(" ", x.name, shape, dt)

    print("Outputs:")
    for x in model.graph.output:
        try:
            shape = model.get_tensor_shape(x.name)
        except Exception:
            shape = None
        try:
            dt = model.get_tensor_datatype(x.name)
        except Exception:
            dt = None
        print(" ", x.name, shape, dt)


def print_graph_report(onnx_path, title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print("ONNX:", onnx_path)

    if not Path(onnx_path).is_file():
        print("[missing file]")
        return None

    stats = graph_stats(onnx_path)

    print_model_io(onnx_path)

    print("\nBasic stats:")
    for k in [
        "node_count",
        "initializer_count",
        "value_info_count",
        "zero_dim_tensor_count",
        "scalar_add_mul_count",
        "multithreshold_count",
    ]:
        print(f"  {k:24s}: {stats[k]}")

    print("\nInteresting op counts:")
    for k, v in stats["interesting_op_counts"].items():
        print(f"  {k:24s}: {v}")

    print("\nQuant-like op counts:")
    if len(stats["quant_like_counts"]) == 0:
        print("  None")
    else:
        for k, v in stats["quant_like_counts"].items():
            print(f"  {k:24s}: {v}")

    print("\nZero-dim tensors:")
    if len(stats["bad_shape_tensors"]) == 0:
        print("  None")
    else:
        for name, shape in stats["bad_shape_tensors"][:30]:
            print(" ", name, shape)
        if len(stats["bad_shape_tensors"]) > 30:
            print(f"  ... {len(stats['bad_shape_tensors']) - 30} more")

    print("\nScalar Add/Mul candidates:")
    if len(stats["scalar_add_mul_names"]) == 0:
        print("  None")
    else:
        for name in stats["scalar_add_mul_names"][:30]:
            print(" ", name)
        if len(stats["scalar_add_mul_names"]) > 30:
            print(f"  ... {len(stats['scalar_add_mul_names']) - 30} more")

    return stats


def compare_graph_stats(before_path, after_path, before_name="before", after_name="after"):
    before = graph_stats(before_path)
    after = graph_stats(after_path)

    keys = [
        "node_count",
        "initializer_count",
        "value_info_count",
        "zero_dim_tensor_count",
        "scalar_add_mul_count",
        "multithreshold_count",
    ]

    rows = []
    for k in keys:
        rows.append({
            "metric": k,
            before_name: before[k],
            after_name: after[k],
            "delta": after[k] - before[k],
        })

    all_ops = sorted(set(before["interesting_op_counts"].keys()) | set(after["interesting_op_counts"].keys()))
    for op in all_ops:
        b = before["interesting_op_counts"].get(op, 0)
        a = after["interesting_op_counts"].get(op, 0)
        rows.append({
            "metric": f"op::{op}",
            before_name: b,
            after_name: a,
            "delta": a - b,
        })

    if HAS_PANDAS:
        df = pd.DataFrame(rows)
        display(df)
    else:
        for r in rows:
            print(r)

    return before, after

## 3. Transformation 函数

下面这些就是你原始脚本里真正影响 graph 的步骤。

每个函数后面都会单独保存一个 ONNX 文件，所以你可以 A/B 对比：

```text
clean.onnx
clean_shaped.onnx
clean_shaped_fixed.onnx
finn.onnx
finn_streamlined.onnx
```

In [5]:

# =========================================================
# 4. Transform helpers
# =========================================================

def prepare_qonnx(raw_path: Path, clean_path: Path):
    """
    Raw QONNX -> cleaned QONNX + datatype inference.
    """
    cleanup(str(raw_path), out_file=str(clean_path))

    qmodel = ModelWrapper(str(clean_path))
    qmodel = qmodel.transform(InferDataTypes())
    qmodel.save(str(clean_path))

    print(f"Prepared QONNX saved to: {clean_path}")


def materialize_exec_shapes(onnx_in_path, onnx_out_path, dummy_input_np, verbose=True):
    """
    Run the ONNX graph once and write observed runtime shapes back to value_info.

    This is useful when cleanup/infer-shape leaves [0, ...] or incomplete shapes.
    """
    model = ModelWrapper(str(onnx_in_path))
    input_name = model.graph.input[0].name

    exec_ctx = oxe.execute_onnx(
        model,
        {input_name: dummy_input_np},
        return_full_exec_context=True,
    )

    updated = 0
    skipped = 0
    repaired = []

    for tensor_name, value in exec_ctx.items():
        if not hasattr(value, "shape"):
            skipped += 1
            continue

        real_shape = _normalize_shape(list(value.shape))

        if real_shape is None or len(real_shape) == 0:
            skipped += 1
            continue

        try:
            old_shape = _normalize_shape(model.get_tensor_shape(tensor_name))
        except Exception:
            old_shape = None

        try:
            model.set_tensor_shape(tensor_name, real_shape)
            updated += 1

            if old_shape != real_shape:
                repaired.append((tensor_name, old_shape, real_shape))

        except Exception:
            skipped += 1

    try:
        model = model.transform(InferDataTypes())
    except Exception as e:
        if verbose:
            print(f"[materialize_exec_shapes] InferDataTypes skipped due to: {e}")

    # Important: do not run InferShapes here, otherwise some real shapes may become bad again.
    model.save(str(onnx_out_path))

    if verbose:
        print(f"[materialize_exec_shapes] updated={updated}, skipped={skipped}")
        print(f"[materialize_exec_shapes] repaired_count={len(repaired)}")
        print(f"[materialize_exec_shapes] saved to: {onnx_out_path}")

        if len(repaired) > 0:
            print("\n=== Materialized / repaired tensor shapes ===")
            for name, old_shape, new_shape in repaired[:80]:
                print(f"{name}: {old_shape} -> {new_shape}")
            if len(repaired) > 80:
                print(f"... {len(repaired) - 80} more")


def replace_zero_dims_with_unknown(onnx_in_path, onnx_out_path):
    """
    Replace ONNX dim_value == 0 with symbolic unknown dimensions.

    This prevents later passes from treating 0 as a real tensor dimension.
    """
    model = onnx.load(str(onnx_in_path))

    all_vis = list(model.graph.input) + list(model.graph.output) + list(model.graph.value_info)
    changed = 0

    for vi in all_vis:
        tt = vi.type.tensor_type

        if not tt.HasField("shape"):
            continue

        dims = tt.shape.dim
        local_changed = False

        for d in dims:
            if d.HasField("dim_value") and d.dim_value == 0:
                d.ClearField("dim_value")
                d.dim_param = "unk"
                local_changed = True

        if local_changed:
            changed += 1

    onnx.save(model, str(onnx_out_path))
    print(f"[replace_zero_dims_with_unknown] changed={changed}, saved to: {onnx_out_path}")


def check_relu_quant_compatibility(qonnx_path: Path):
    """
    Check Relu -> Quant compatibility for FINN.

    For QuantReLU, the quantizer should normally be unsigned and non-narrow.
    """
    qmodel = ModelWrapper(str(qonnx_path))
    bad_nodes = []

    print("\n=== Checking Relu -> Quant compatibility for FINN ===")

    for node in qmodel.graph.node:
        if node.op_type != "Quant":
            continue

        preds = qmodel.find_direct_predecessors(node)

        if preds is None or len(preds) == 0:
            continue

        prev = preds[0]

        if prev.op_type != "Relu":
            continue

        qop = getCustomOp(node)
        signed = qop.get_nodeattr("signed")
        narrow = qop.get_nodeattr("narrow")

        print(f"Node: {node.name if node.name else '<unnamed>'}")
        print(f"  signed = {signed}")
        print(f"  narrow = {narrow}")

        if signed or narrow:
            bad_nodes.append(node.name if node.name else "<unnamed>")

    if len(bad_nodes) > 0:
        raise RuntimeError(
            "Found FINN-incompatible Relu->Quant node(s): "
            + ", ".join(bad_nodes)
            + "\nPlease use unsigned / non-narrow activation quant after ReLU."
        )

    print("All Relu -> Quant nodes look compatible.")


def convert_qonnx_to_finn(clean_qonnx_path: Path, finn_onnx_path: Path):
    model = ModelWrapper(str(clean_qonnx_path))

    model = model.transform(ConvertQONNXtoFINN())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())

    try:
        model = model.transform(InferDataTypes())
    except Exception as e:
        print(f"[WARN] InferDataTypes on FINN model skipped: {e}")

    model.save(str(finn_onnx_path))
    print(f"FINN-ONNX saved to: {finn_onnx_path}")


def extra_streamline_bias_scale_chains(in_path, out_path, max_iter=20):
    """
    Repeatedly apply scalar/bias absorption passes around MultiThreshold.

    Evidence that this helped:
      - node_count decreases
      - scalar_add_mul_count decreases
      - standalone Add/Mul around threshold paths decrease
      - QONNX-vs-FINN verification still passes
    """
    model = ModelWrapper(str(in_path))

    prev_num_nodes = -1
    it = 0

    while prev_num_nodes != len(model.graph.node) and it < max_iter:
        prev_num_nodes = len(model.graph.node)
        it += 1

        model = model.transform(DuplicateScalarMulAfterFork())
        model = model.transform(AbsorbMulAddIntoMultiThreshold())
        model = model.transform(AbsorbAddIntoMultiThreshold())
        model = model.transform(AbsorbMulIntoMultiThreshold())
        model = model.transform(AbsorbSignBiasIntoMultiThreshold())

        print(f"[streamline iter {it}] nodes: {prev_num_nodes} -> {len(model.graph.node)}")

    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    model.save(str(out_path))
    print(f"Saved extra-streamlined model to: {out_path}")

## 4. Verification 函数

这里的目的不是训练或测数据集精度，而是确认：

```text
PyTorch output ≈ QONNX output
QONNX output ≈ FINN-ONNX output
```

如果 graph cleanup 后数值变了，说明这个 cleanup 不能直接用于最终部署。

In [6]:

# =========================================================
# 5. Verification helpers
# =========================================================

def run_onnx_logits(onnx_path, input_np, tag=""):
    model = ModelWrapper(str(onnx_path))
    input_name = model.graph.input[0].name

    out_dict = oxe.execute_onnx(
        model,
        {input_name: input_np},
    )

    _, raw = pick_logits_output_from_onnx_outputs(
        out_dict,
        target_num_classes=NUM_CLASSES,
        tag=tag,
    )

    logits = logits_to_2d_np(raw)
    return raw, logits


def verify_pytorch_vs_qonnx(model, qonnx_path: Path, dummy_input):
    model.eval()

    with torch.no_grad():
        pyt_raw_out = qt_value(model(dummy_input))
        pyt_out = logits_to_2d_torch(pyt_raw_out).cpu().numpy()

    qonnx_raw_out, qonnx_out = run_onnx_logits(
        qonnx_path,
        dummy_input.detach().cpu().numpy(),
        tag="QONNX",
    )

    diff = np.abs(pyt_out - qonnx_out)
    allclose = np.allclose(pyt_out, qonnx_out, atol=1e-5, rtol=1e-5)

    print("\n=== PyTorch vs QONNX ===")
    print("PyTorch raw output shape :", tuple(pyt_raw_out.shape))
    print("PyTorch logits shape     :", pyt_out.shape)
    print("QONNX raw output shape   :", np.asarray(qonnx_raw_out).shape)
    print("QONNX logits shape       :", qonnx_out.shape)
    print("max abs diff             :", diff.max())
    print("mean abs diff            :", diff.mean())
    print("argmax match             :", (pyt_out.argmax(axis=1) == qonnx_out.argmax(axis=1)).mean())
    print("out_a range              :", pyt_out.min(), pyt_out.max())
    print("out_b range              :", qonnx_out.min(), qonnx_out.max())
    print("Allclose                 :", allclose)

    return {
        "allclose": bool(allclose),
        "max_abs_diff": float(diff.max()),
        "mean_abs_diff": float(diff.mean()),
        "argmax_match": float((pyt_out.argmax(axis=1) == qonnx_out.argmax(axis=1)).mean()),
    }


def verify_onnx_vs_onnx(path_a: Path, path_b: Path, dummy_input_np, tag_a="A", tag_b="B"):
    raw_a, out_a = run_onnx_logits(path_a, dummy_input_np, tag=tag_a)
    raw_b, out_b = run_onnx_logits(path_b, dummy_input_np, tag=tag_b)

    diff = np.abs(out_a - out_b)
    allclose = np.allclose(out_a, out_b, atol=1e-5, rtol=1e-5)

    print(f"\n=== {tag_a} vs {tag_b} ===")
    print(f"{tag_a} raw output shape :", np.asarray(raw_a).shape)
    print(f"{tag_b} raw output shape :", np.asarray(raw_b).shape)
    print(f"{tag_a} logits shape     :", out_a.shape)
    print(f"{tag_b} logits shape     :", out_b.shape)
    print("max abs diff           :", diff.max())
    print("mean abs diff          :", diff.mean())
    print("argmax match           :", (out_a.argmax(axis=1) == out_b.argmax(axis=1)).mean())
    print("Allclose               :", allclose)

    return {
        "allclose": bool(allclose),
        "max_abs_diff": float(diff.max()),
        "mean_abs_diff": float(diff.mean()),
        "argmax_match": float((out_a.argmax(axis=1) == out_b.argmax(axis=1)).mean()),
    }

## 5. 加载 PyTorch/Brevitas 模型

这里会检查 checkpoint 和当前 `newnet.py` 是否匹配。

如果这里报 serious missing/unexpected keys，说明你当前模型定义和 checkpoint 不是同一版。

In [7]:

# =========================================================
# 6. Load PyTorch model from checkpoint
# =========================================================

def build_model_from_checkpoint(ckpt_path: Path):
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "model_state_dict" not in ckpt or "model_config" not in ckpt:
        raise RuntimeError("Checkpoint missing model_state_dict or model_config")

    cfg = ckpt["model_config"]

    model = ECG5000FullQuantTCN(
        num_classes=cfg.get("num_classes", NUM_CLASSES),
        seq_len=cfg.get("seq_len", SEQ_LEN),
        num_inputs=cfg.get("num_inputs", IN_CH),
        num_channels=tuple(cfg.get("num_channels", (16, 16, 32))),
        kernel_size=cfg.get("kernel_size", 3),
        dropout=cfg.get("dropout", 0.0),
        causal=cfg.get("causal", True),
    )

    raw_state_dict = ckpt["model_state_dict"]

    cleaned_state_dict = {}
    for k, v in raw_state_dict.items():
        if k.startswith("module."):
            cleaned_state_dict[k[7:]] = v
        else:
            cleaned_state_dict[k] = v

    load_result = model.load_state_dict(cleaned_state_dict, strict=False)

    missing = list(load_result.missing_keys)
    unexpected = list(load_result.unexpected_keys)

    print("Checkpoint loaded.")
    print("Model config    :", cfg)
    print("Missing keys    :", missing)
    print("Unexpected keys :", unexpected)

    serious_missing = [k for k in missing if not k.endswith(".value")]
    serious_unexpected = [k for k in unexpected if not k.endswith(".value")]

    if len(serious_missing) > 0 or len(serious_unexpected) > 0:
        raise RuntimeError(
            "Checkpoint and current model structure do not match.\n"
            f"Serious missing keys   : {serious_missing}\n"
            f"Serious unexpected keys: {serious_unexpected}"
        )

    return model


torch.manual_seed(0)
np.random.seed(0)

model = build_model_from_checkpoint(CHECKPOINT_PATH).to(DEVICE)
model.eval()

dummy_input = torch.randn(BATCH_SIZE, IN_CH, SEQ_LEN, 1, device=DEVICE)
dummy_input_np = dummy_input.detach().cpu().numpy()

print("dummy_input shape:", tuple(dummy_input.shape))

in_ch = 1
out_ch = 16
in_ch = 16
out_ch = 16
in_ch = 16
out_ch = 16
Checkpoint loaded.
Model config    : {'num_classes': 5, 'seq_len': 140, 'num_inputs': 1, 'num_channels': (16, 16, 16), 'kernel_size': 3, 'dropout': 0.1, 'causal': True, 'weight_bits': 4, 'act_bits': 4, 'input_bits': 4, 'export_all_time_logits': False, 'expected_output_shape': '[N, 5, 1, 1]', 'loaded_from': 'C:\\Users\\12209\\Desktop\\Thesis\\pytorch-tcn\\new_version\\best_qtcn_model_last_step.pth'}
Missing keys    : []
Unexpected keys : []
dummy_input shape: (1, 1, 140, 1)


## 6. 检查 PyTorch 原始输出

B-scheme 的目标输出必须是：

```text
[1, 5, 1, 1]
```

如果这里不是这个 shape，说明你的 `newnet.py` 或 checkpoint 还不是 temporal classifier 版本。

In [8]:

# =========================================================
# 7. Check PyTorch output shape before export
# =========================================================

with torch.no_grad():
    pyt_raw_out = qt_value(model(dummy_input))
    pyt_logits = logits_to_2d_torch(pyt_raw_out)

print("\n=== PyTorch raw output before export ===")
print("raw output shape       :", tuple(pyt_raw_out.shape))
print("converted logits shape :", tuple(pyt_logits.shape))
print("raw output range       :", float(pyt_raw_out.min()), float(pyt_raw_out.max()))
print("last-step logits range :", float(pyt_logits.min()), float(pyt_logits.max()))
print("last-step pred         :", pyt_logits.argmax(dim=1).detach().cpu().numpy())

expected_raw_shape = (BATCH_SIZE, NUM_CLASSES, 1, 1)
if tuple(pyt_raw_out.shape) != expected_raw_shape:
    raise RuntimeError(
        "B-scheme model output shape is wrong before export.\n"
        f"Expected PyTorch raw output shape: {expected_raw_shape}\n"
        f"Got: {tuple(pyt_raw_out.shape)}\n"
        "This means newnet.py is not using the temporal classifier that directly outputs [N, 5, 1, 1], "
        "or the checkpoint/model class is not the B version."
    )

out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(0.5000)
res scale: tensor(0.5000)
out bit_width: tensor(4.)
res bit_width: tensor(4.)

=== PyTorch raw output before export ===
raw output shape       : (1, 5, 1, 1)
converted logits shape : (1, 5)
raw output range       : -0.884765625 0.30615234375
last-step logits range : -0.884765625 0.30615234375
last-step pred         : [2]


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)


## 7. Step A：导出 Raw QONNX

这一阶段主要看：

- raw graph 的 op 结构；
- 是否已经有 Quant / Trunc 等 QONNX 量化节点；
- 输出 shape 是否还是 `[1, 5, 1, 1]`；
- 是否存在 zero-dim tensor。

如果 Raw QONNX 已经很干净，后面 clean/materialize 变化会较小；如果这里有很多异常 shape，后面步骤应该能看到改善。

In [9]:

# =========================================================
# 8. Export raw QONNX
# =========================================================

def export_model_to_qonnx(model, dummy_input, out_path: Path):
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)

    export_qonnx(
        model,
        dummy_input,
        str(out_path),
    )

    print(f"Raw QONNX saved to: {out_path}")


if FORCE_REEXPORT or not RAW_QONNX_PATH.is_file():
    export_model_to_qonnx(model, dummy_input, RAW_QONNX_PATH)
else:
    print("Reuse existing:", RAW_QONNX_PATH)

raw_stats = print_graph_report(RAW_QONNX_PATH, "STEP A / Raw QONNX")

out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(0.5000)
res scale: tensor(0.5000)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(0.5000)
res scale: tensor(0.5000)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(0.5000)
res scale: tensor(0.5000)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
Raw QONNX saved to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwi

## 8. Step B：QONNX cleanup + datatype inference

这里测试 `cleanup(...) + InferDataTypes()` 是否有帮助。

你应该重点看：

- node count 是否减少；
- op histogram 是否更规整；
- tensor datatype 是否更完整；
- zero-dim 是否变化。

注意：cleanup 不一定能解决所有 shape 问题，所以后面还有 materialization。

In [10]:

# =========================================================
# 9. Prepare QONNX: cleanup + datatype inference
# =========================================================

if FORCE_REEXPORT or not CLEAN_QONNX_PATH.is_file():
    prepare_qonnx(RAW_QONNX_PATH, CLEAN_QONNX_PATH)
else:
    print("Reuse existing:", CLEAN_QONNX_PATH)

clean_stats = print_graph_report(CLEAN_QONNX_PATH, "STEP B / Clean QONNX")

print("\n=== Compare: Raw -> Clean ===")
compare_graph_stats(RAW_QONNX_PATH, CLEAN_QONNX_PATH, "raw", "clean")

Prepared QONNX saved to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean.onnx

STEP B / Clean QONNX
ONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean.onnx
Inputs:
  global_in [1, 1, 140, 1] FLOAT32
Outputs:
  global_out [1, 5, 1, 1] SCALEDINT<32>

Basic stats:
  node_count              : 52
  initializer_count       : 115
  value_info_count        : 166
  zero_dim_tensor_count   : 0
  scalar_add_mul_count    : 0
  multithreshold_count    : 0

Interesting op counts:
  Quant                   : 29
  Conv                    : 8
  Pad                     : 6
  Add                     : 3
  Relu                    : 6

Quant-like op counts:
  Quant                   : 29

Zero-dim tensors:
  None

Scalar Add/Mul candidates:
  None

=== Compare: Raw -> Clean ===


,metric,raw,clean,delta
0,node_count,94,52,-42
1,initializer_count,81,115,34
2,value_info_count,102,166,64
3,zero_dim_tensor_count,30,0,-30
4,scalar_add_mul_count,0,0,0
5,multithreshold_count,0,0,0
6,op::Add,3,3,0
7,op::Conv,8,8,0
8,op::Pad,6,6,0
9,op::Quant,29,29,0


(OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx.onnx'),
              ('exists', True),
              ('node_count', 94),
              ('initializer_count', 81),
              ('value_info_count', 102),
              ('zero_dim_tensor_count', 30),
              ('scalar_add_mul_count', 0),
              ('multithreshold_count', 0),
              ('quant_like_counts', {'Quant': 29}),
              ('interesting_op_counts',
               {'Quant': 29,
                'Conv': 8,
                'Transpose': 6,
                'Reshape': 12,
                'Pad': 6,
                'Add': 3,
                'Relu': 6}),
              ('bad_shape_tensors',
               [('/backbone/backbone.0/act1/act_quant/export_handler/Quant_output_0',
                 [0, 0, 0, 0]),
                ('/backbone/backbone.0/conv1/conv/bias_quant/export_handler/Quant_output_0',
                 [0]),
      

## 9. Step C：materialize_exec_shapes

这是你脚本里最值得验证的一步。

它的理论作用是：

```text
执行一次 clean QONNX -> 拿到真实中间 tensor shape -> 写回 graph
```

判断它有没有用，主要看：

- `zero_dim_tensor_count` 是否下降；
- repaired tensor shapes 列表是否有大量 `[0, ...] -> [real, ...]`；
- 后续 ConvertQONNXtoFINN 是否不再因为 shape 出错。

In [11]:

# =========================================================
# 10. Materialize execution shapes
# =========================================================

if FORCE_REEXPORT or not CLEAN_QONNX_SHAPED_PATH.is_file():
    materialize_exec_shapes(
        CLEAN_QONNX_PATH,
        CLEAN_QONNX_SHAPED_PATH,
        dummy_input_np,
        verbose=True,
    )
else:
    print("Reuse existing:", CLEAN_QONNX_SHAPED_PATH)

shaped_stats = print_graph_report(CLEAN_QONNX_SHAPED_PATH, "STEP C / Clean QONNX + materialized shapes")

print("\n=== Compare: Clean -> Materialized ===")
compare_graph_stats(CLEAN_QONNX_PATH, CLEAN_QONNX_SHAPED_PATH, "clean", "materialized")

[materialize_exec_shapes] updated=75, skipped=93
[materialize_exec_shapes] repaired_count=0
[materialize_exec_shapes] saved to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped.onnx

STEP C / Clean QONNX + materialized shapes
ONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped.onnx
Inputs:
  global_in [1, 1, 140, 1] FLOAT32
Outputs:
  global_out [1, 5, 1, 1] SCALEDINT<32>

Basic stats:
  node_count              : 52
  initializer_count       : 115
  value_info_count        : 166
  zero_dim_tensor_count   : 0
  scalar_add_mul_count    : 0
  multithreshold_count    : 0

Interesting op counts:
  Quant                   : 29
  Conv                    : 8
  Pad                     : 6
  Add                     : 3
  Relu                    : 6

Quant-like op counts:
  Quant                   : 29

Zero-dim tensors:
  None

Scalar Add/Mul candidates:


,metric,clean,materialized,delta
0,node_count,52,52,0
1,initializer_count,115,115,0
2,value_info_count,166,166,0
3,zero_dim_tensor_count,0,0,0
4,scalar_add_mul_count,0,0,0
5,multithreshold_count,0,0,0
6,op::Add,3,3,0
7,op::Conv,8,8,0
8,op::Pad,6,6,0
9,op::Quant,29,29,0


(OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean.onnx'),
              ('exists', True),
              ('node_count', 52),
              ('initializer_count', 115),
              ('value_info_count', 166),
              ('zero_dim_tensor_count', 0),
              ('scalar_add_mul_count', 0),
              ('multithreshold_count', 0),
              ('quant_like_counts', {'Quant': 29}),
              ('interesting_op_counts',
               {'Quant': 29, 'Conv': 8, 'Pad': 6, 'Add': 3, 'Relu': 6}),
              ('bad_shape_tensors', []),
              ('scalar_add_mul_names', []),
              ('op_counts',
               {'Add': 3, 'Conv': 8, 'Pad': 6, 'Quant': 29, 'Relu': 6})]),
 OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped.onnx'),
              ('exists', True),
           

## 10. Step D：replace_zero_dims_with_unknown

这一阶段比较特殊：它不一定减少 node，也不一定改变算子结构。

它的作用是：

```text
把 ONNX value_info 里 dim_value == 0 的维度改成 symbolic unknown
```

判断它有没有用：

- `zero_dim_tensor_count` 应该变成 0 或减少；
- 数值验证应该不变；
- 后续 FINN conversion 不应该因为 zero shape 崩。

In [12]:

# =========================================================
# 11. Replace zero dims with unknown
# =========================================================

if FORCE_REEXPORT or not CLEAN_QONNX_SHAPED_FIXED_PATH.is_file():
    replace_zero_dims_with_unknown(
        CLEAN_QONNX_SHAPED_PATH,
        CLEAN_QONNX_SHAPED_FIXED_PATH,
    )
else:
    print("Reuse existing:", CLEAN_QONNX_SHAPED_FIXED_PATH)

fixed_stats = print_graph_report(CLEAN_QONNX_SHAPED_FIXED_PATH, "STEP D / Clean QONNX + materialized + zero-fixed")

print("\n=== Compare: Materialized -> Zero-fixed ===")
compare_graph_stats(CLEAN_QONNX_SHAPED_PATH, CLEAN_QONNX_SHAPED_FIXED_PATH, "materialized", "zero_fixed")

[replace_zero_dims_with_unknown] changed=0, saved to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx

STEP D / Clean QONNX + materialized + zero-fixed
ONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx
Inputs:
  global_in [1, 1, 140, 1] FLOAT32
Outputs:
  global_out [1, 5, 1, 1] SCALEDINT<32>

Basic stats:
  node_count              : 52
  initializer_count       : 115
  value_info_count        : 166
  zero_dim_tensor_count   : 0
  scalar_add_mul_count    : 0
  multithreshold_count    : 0

Interesting op counts:
  Quant                   : 29
  Conv                    : 8
  Pad                     : 6
  Add                     : 3
  Relu                    : 6

Quant-like op counts:
  Quant                   : 29

Zero-dim tensors:
  None

Scalar Add/Mul candidates:
  None

=== Compare: Materialized -> Zero-fixed ===


,metric,materialized,zero_fixed,delta
0,node_count,52,52,0
1,initializer_count,115,115,0
2,value_info_count,166,166,0
3,zero_dim_tensor_count,0,0,0
4,scalar_add_mul_count,0,0,0
5,multithreshold_count,0,0,0
6,op::Add,3,3,0
7,op::Conv,8,8,0
8,op::Pad,6,6,0
9,op::Quant,29,29,0


(OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped.onnx'),
              ('exists', True),
              ('node_count', 52),
              ('initializer_count', 115),
              ('value_info_count', 166),
              ('zero_dim_tensor_count', 0),
              ('scalar_add_mul_count', 0),
              ('multithreshold_count', 0),
              ('quant_like_counts', {'Quant': 29}),
              ('interesting_op_counts',
               {'Quant': 29, 'Conv': 8, 'Pad': 6, 'Add': 3, 'Relu': 6}),
              ('bad_shape_tensors', []),
              ('scalar_add_mul_names', []),
              ('op_counts',
               {'Add': 3, 'Conv': 8, 'Pad': 6, 'Quant': 29, 'Relu': 6})]),
 OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx'),
              ('exists', True)

## 11. Step E：验证 PyTorch vs Clean/Fixed QONNX

如果这里误差很大，说明前面的 graph cleanup 已经改变了模型语义，不能继续用于硬件生成。

正常情况下：

- `max abs diff` 很小；
- `argmax match = 1.0`。

In [13]:

# =========================================================
# 12. Verify PyTorch vs fixed clean QONNX
# =========================================================

pt_vs_qonnx_result = verify_pytorch_vs_qonnx(
    model,
    CLEAN_QONNX_SHAPED_FIXED_PATH,
    dummy_input,
)

pt_vs_qonnx_result

out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(1.)
res scale: tensor(1.)
out bit_width: tensor(4.)
res bit_width: tensor(4.)
out scale: tensor(0.5000)
res scale: tensor(0.5000)
out bit_width: tensor(4.)
res bit_width: tensor(4.)

=== Available ONNX outputs: QONNX ===
global_out: shape=(1, 5, 1, 1)
Chosen logits output: global_out, raw shape=(1, 5, 1, 1)
Chosen logits converted shape=(1, 5)

=== PyTorch vs QONNX ===
PyTorch raw output shape : (1, 5, 1, 1)
PyTorch logits shape     : (1, 5)
QONNX raw output shape   : (1, 5, 1, 1)
QONNX logits shape       : (1, 5)
max abs diff             : 0.0
mean abs diff            : 0.0
argmax match             : 1.0
out_a range              : -0.8847656 0.30615234
out_b range              : -0.8847656 0.30615234
Allclose                 : True


{'allclose': True,
 'max_abs_diff': 0.0,
 'mean_abs_diff': 0.0,
 'argmax_match': 1.0}

## 12. Step F：检查 ReLU -> Quant 兼容性

这个步骤不会改变 graph，它是一个 guard/check。

如果有 ReLU 后面接 signed 或 narrow Quant，FINN 可能不接受，或者语义上和 ReLU 的非负输出不一致。

它的论文表达可以是：

```text
A compatibility check was applied to ensure that ReLU-following activation quantizers use unsigned non-narrow quantization.
```

In [14]:

# =========================================================
# 13. Check ReLU -> Quant compatibility
# =========================================================

check_relu_quant_compatibility(CLEAN_QONNX_SHAPED_FIXED_PATH)


=== Checking Relu -> Quant compatibility for FINN ===
Node: Quant_18
  signed = 0
  narrow = 0
Node: Quant_20
  signed = 0
  narrow = 0
Node: Quant_22
  signed = 0
  narrow = 0
Node: Quant_24
  signed = 0
  narrow = 0
Node: Quant_26
  signed = 0
  narrow = 0
Node: Quant_28
  signed = 0
  narrow = 0
All Relu -> Quant nodes look compatible.


## 13. Step G：Convert QONNX to FINN-ONNX

这里开始从 QONNX 表达变成 FINN custom op 表达。

重点看：

- Quant 是否变成 MultiThreshold；
- MatMul/Conv/Im2Col 是否变成 FINN-friendly pattern；
- 输出 shape 是否还正确；
- zero-dim 是否仍然为 0。

In [15]:

# =========================================================
# 14. Convert QONNX to FINN-ONNX
# =========================================================

if FORCE_REEXPORT or not FINN_ONNX_PATH.is_file():
    convert_qonnx_to_finn(CLEAN_QONNX_SHAPED_FIXED_PATH, FINN_ONNX_PATH)
else:
    print("Reuse existing:", FINN_ONNX_PATH)

finn_stats = print_graph_report(FINN_ONNX_PATH, "STEP G / FINN-ONNX")

print("\n=== Compare: Clean fixed QONNX -> FINN-ONNX ===")
compare_graph_stats(CLEAN_QONNX_SHAPED_FIXED_PATH, FINN_ONNX_PATH, "qonnx_fixed", "finn")

FINN-ONNX saved to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn.onnx

STEP G / FINN-ONNX
ONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn.onnx
Inputs:
  global_in [1, 1, 140, 1] FLOAT32
Outputs:
  global_out [1, 5, 1, 1] FLOAT32

Basic stats:
  node_count              : 76
  initializer_count       : 79
  value_info_count        : 154
  zero_dim_tensor_count   : 0
  scalar_add_mul_count    : 30
  multithreshold_count    : 13

Interesting op counts:
  MultiThreshold          : 13
  Conv                    : 8
  Pad                     : 6
  Mul                     : 23
  Add                     : 18
  Div                     : 8

Quant-like op counts:
  MultiThreshold          : 13

Zero-dim tensors:
  None

Scalar Add/Mul candidates:
  Add_0
  Mul_0
  Mul_1
  Mul_2
  Mul_3
  Add_3
  Mul_4
  Mul_5
  Mul_6
  Add_5
  Add_7
  Mul_7
  Mul_8
  Mul_9
  Mul_10
  Mul_11
  A

,metric,qonnx_fixed,finn,delta
0,node_count,52,76,24
1,initializer_count,115,79,-36
2,value_info_count,166,154,-12
3,zero_dim_tensor_count,0,0,0
4,scalar_add_mul_count,0,30,30
5,multithreshold_count,0,13,13
6,op::Add,3,18,15
7,op::Conv,8,8,0
8,op::Div,0,8,8
9,op::Mul,0,23,23


(OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_qonnx_clean_shaped_fixed.onnx'),
              ('exists', True),
              ('node_count', 52),
              ('initializer_count', 115),
              ('value_info_count', 166),
              ('zero_dim_tensor_count', 0),
              ('scalar_add_mul_count', 0),
              ('multithreshold_count', 0),
              ('quant_like_counts', {'Quant': 29}),
              ('interesting_op_counts',
               {'Quant': 29, 'Conv': 8, 'Pad': 6, 'Add': 3, 'Relu': 6}),
              ('bad_shape_tensors', []),
              ('scalar_add_mul_names', []),
              ('op_counts',
               {'Add': 3, 'Conv': 8, 'Pad': 6, 'Quant': 29, 'Relu': 6})]),
 OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn.onnx'),
              ('exists', True),
            

## 14. Step H：extra_streamline_bias_scale_chains

这里测试 scalar scale/bias absorption 是否真的减少了额外算子。

判断它是否有帮助，看下面几个指标：

- `node_count` 是否下降；
- `scalar_add_mul_count` 是否下降；
- `Mul/Add` 是否被吸收到 `MultiThreshold`；
- QONNX vs FINN-streamlined 的输出是否仍然一致。

如果节点没有减少，但后续 build 更顺，仍然可以说明它对 graph canonicalization 有帮助；不过论文里要写得克制。

In [16]:

# =========================================================
# 15. Extra streamline scalar/bias chains
# =========================================================

if FORCE_REEXPORT or not FINN_ONNX_STREAMLINED_PATH.is_file():
    extra_streamline_bias_scale_chains(FINN_ONNX_PATH, FINN_ONNX_STREAMLINED_PATH)
else:
    print("Reuse existing:", FINN_ONNX_STREAMLINED_PATH)

stream_stats = print_graph_report(FINN_ONNX_STREAMLINED_PATH, "STEP H / FINN-ONNX extra-streamlined")

print("\n=== Compare: FINN -> FINN streamlined ===")
compare_graph_stats(FINN_ONNX_PATH, FINN_ONNX_STREAMLINED_PATH, "finn", "streamlined")

[streamline iter 1] nodes: 76 -> 62
[streamline iter 2] nodes: 62 -> 55
[streamline iter 3] nodes: 55 -> 55
Saved extra-streamlined model to: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn_streamlined.onnx

STEP H / FINN-ONNX extra-streamlined
ONNX: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn_streamlined.onnx
Inputs:
  global_in [1, 1, 140, 1] FLOAT32
Outputs:
  global_out [1, 5, 1, 1] FLOAT32

Basic stats:
  node_count              : 55
  initializer_count       : 58
  value_info_count        : 112
  zero_dim_tensor_count   : 0
  scalar_add_mul_count    : 16
  multithreshold_count    : 13

Interesting op counts:
  MultiThreshold          : 13
  Conv                    : 8
  Pad                     : 6
  Mul                     : 16
  Add                     : 4
  Div                     : 8

Quant-like op counts:
  MultiThreshold          : 13

Zero-dim tensors:
  Non

/home/slowman/Desktop/project/Thesis/finn/src/finn/transformation/streamline/absorb.py:130: UserWarning: Threshold or add bias not constant, skipping
  warnings.warn("Threshold or add bias not constant, skipping")


,metric,finn,streamlined,delta
0,node_count,76,55,-21
1,initializer_count,79,58,-21
2,value_info_count,154,112,-42
3,zero_dim_tensor_count,0,0,0
4,scalar_add_mul_count,30,16,-14
5,multithreshold_count,13,13,0
6,op::Add,18,4,-14
7,op::Conv,8,8,0
8,op::Div,8,8,0
9,op::Mul,23,16,-7


(OrderedDict([('path',
               '/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_pointwise_qtcn_finn.onnx'),
              ('exists', True),
              ('node_count', 76),
              ('initializer_count', 79),
              ('value_info_count', 154),
              ('zero_dim_tensor_count', 0),
              ('scalar_add_mul_count', 30),
              ('multithreshold_count', 13),
              ('quant_like_counts', {'MultiThreshold': 13}),
              ('interesting_op_counts',
               {'MultiThreshold': 13,
                'Conv': 8,
                'Pad': 6,
                'Mul': 23,
                'Add': 18,
                'Div': 8}),
              ('bad_shape_tensors', []),
              ('scalar_add_mul_names',
               ['Add_0',
                'Mul_0',
                'Mul_1',
                'Mul_2',
                'Mul_3',
                'Add_3',
                'Mul_4',
                'Mul_5',
              

## 15. 检查 MultiThreshold 和最终 classifier 映射

这一步帮助你确认两个事情：

1. activation quantization 是否真的变成了 `MultiThreshold`；
2. final temporal classifier 是否变成类似 `MW = channels * seq_len`, `MH = num_classes` 的结构。

如果 final classifier 是 `[16,16,16]`，理论上最后可能是：

```text
MW = 16 * 140 = 2240
MH = 5
```

如果最后通道是 32，则可能是：

```text
MW = 32 * 140 = 4480
MH = 5
```

In [17]:

# =========================================================
# 16. Inspect MultiThreshold and final classifier candidates
# =========================================================

def inspect_multithreshold_nodes(onnx_path: Path, title: str, max_nodes=20):
    model = ModelWrapper(str(onnx_path))

    print(f"\n=== Inspect MultiThreshold: {title} ===")
    mt_nodes = [n for n in model.graph.node if n.op_type == "MultiThreshold"]
    print("MultiThreshold count:", len(mt_nodes))

    if len(mt_nodes) == 0:
        print("No MultiThreshold nodes found.")
        return

    for i, node in enumerate(mt_nodes[:max_nodes]):
        in_name = node.input[0]
        th_name = node.input[1]

        try:
            in_shape = model.get_tensor_shape(in_name)
        except Exception:
            in_shape = None

        th = model.get_initializer(th_name)
        th_shape = None if th is None else th.shape

        attrs = {}
        for a in node.attribute:
            try:
                if a.type == 3:   # STRING
                    attrs[a.name] = a.s.decode("utf-8")
                elif a.type == 2: # INT
                    attrs[a.name] = a.i
                elif a.type == 1: # FLOAT
                    attrs[a.name] = a.f
                else:
                    attrs[a.name] = f"<type {a.type}>"
            except Exception:
                attrs[a.name] = "<unparsed>"

        print(f"\n--- MultiThreshold [{i}] ---")
        print("name      :", node.name if node.name else "<unnamed>")
        print("input     :", in_name)
        print("in_shape  :", in_shape)
        print("threshold :", th_name)
        print("th_shape  :", th_shape)
        print("output    :", list(node.output))
        print("attrs     :", attrs)

    if len(mt_nodes) > max_nodes:
        print(f"\n... {len(mt_nodes) - max_nodes} more MultiThreshold nodes not printed.")


def inspect_final_classifier_candidate(onnx_path: Path, title: str):
    model = ModelWrapper(str(onnx_path))

    print(f"\n=== Final classifier candidate inspection: {title} ===")
    print("ONNX:", onnx_path)

    found = 0

    for n in model.graph.node:
        if "MVAU" not in n.op_type and "MatrixVector" not in n.op_type:
            continue

        try:
            cop = getCustomOp(n)
        except Exception:
            continue

        attrs = {}
        for a in [
            "MW", "MH", "SIMD", "PE", 
            "inputDataType", "weightDataType", "outputDataType", 
            "numInputVectors", "noActivation"
        ]:
            try:
                attrs[a] = cop.get_nodeattr(a)
            except Exception:
                pass

        if attrs.get("MH", None) == NUM_CLASSES:
            tag = "  <-- likely final classifier"
        else:
            tag = ""

        found += 1
        print("\nnode:", n.name if n.name else "<unnamed>", tag)
        print("op_type:", n.op_type)
        print("domain:", n.domain)
        print("inputs:", list(n.input))
        print("outputs:", list(n.output))
        print("attrs:", attrs)

        for t in list(n.input) + list(n.output):
            try:
                print("  tensor", t, "shape", model.get_tensor_shape(t), "dtype", model.get_tensor_datatype(t))
            except Exception:
                pass

    if found == 0:
        print("No MVAU/MatrixVector-like nodes found at this stage.")


inspect_multithreshold_nodes(FINN_ONNX_STREAMLINED_PATH, "FINN-ONNX STREAMLINED")
inspect_final_classifier_candidate(FINN_ONNX_STREAMLINED_PATH, "FINN-ONNX STREAMLINED")


=== Inspect MultiThreshold: FINN-ONNX STREAMLINED ===
MultiThreshold count: 13

--- MultiThreshold [0] ---
name      : MultiThreshold_0
input     : global_in
in_shape  : [1, 1, 140, 1]
threshold : MultiThreshold_0_param0
th_shape  : (1, 15)
output    : ['Add_0_out0']
attrs     : {'out_dtype': 'INT4', 'out_bias': -8.0}

--- MultiThreshold [1] ---
name      : MultiThreshold_1
input     : Div_0_out0
in_shape  : [1, 16, 140, 1]
threshold : MultiThreshold_1_param0
th_shape  : (16, 15)
output    : ['Add_3_out0']
attrs     : {'out_dtype': 'INT4', 'out_bias': -8.0}

--- MultiThreshold [2] ---
name      : MultiThreshold_2
input     : Div_1_out0
in_shape  : [1, 16, 140, 1]
threshold : MultiThreshold_2_param0
th_shape  : (16, 15)
output    : ['MultiThreshold_2_out0']
attrs     : {'out_dtype': 'UINT4'}

--- MultiThreshold [3] ---
name      : MultiThreshold_3
input     : Div_2_out0
in_shape  : [1, 16, 140, 1]
threshold : MultiThreshold_3_param0
th_shape  : (16, 15)
output    : ['Add_5_out0']
attrs

## 16. Step I：验证 Clean QONNX vs FINN-ONNX Streamlined

这一步最关键。

如果 `extra_streamline_bias_scale_chains` 之后输出还能对齐，说明它没有破坏模型语义。

正常期望：

```text
argmax match = 1.0
max abs diff 很小
```

如果 `Allclose=False` 但 argmax match 仍然是 1.0，需要看数值范围和 diff 是不是来自 scale/threshold 表达差异。

In [18]:

# =========================================================
# 17. Verify clean QONNX vs FINN-ONNX streamlined
# =========================================================

qonnx_vs_finn_result = verify_onnx_vs_onnx(
    CLEAN_QONNX_SHAPED_FIXED_PATH,
    FINN_ONNX_STREAMLINED_PATH,
    dummy_input_np,
    tag_a="CLEAN_QONNX_FIXED",
    tag_b="FINN_STREAMLINED",
)

qonnx_vs_finn_result


=== Available ONNX outputs: CLEAN_QONNX_FIXED ===
global_out: shape=(1, 5, 1, 1)
Chosen logits output: global_out, raw shape=(1, 5, 1, 1)
Chosen logits converted shape=(1, 5)

=== Available ONNX outputs: FINN_STREAMLINED ===
global_out: shape=(1, 5, 1, 1)
Chosen logits output: global_out, raw shape=(1, 5, 1, 1)
Chosen logits converted shape=(1, 5)

=== CLEAN_QONNX_FIXED vs FINN_STREAMLINED ===
CLEAN_QONNX_FIXED raw output shape : (1, 5, 1, 1)
FINN_STREAMLINED raw output shape : (1, 5, 1, 1)
CLEAN_QONNX_FIXED logits shape     : (1, 5)
FINN_STREAMLINED logits shape     : (1, 5)
max abs diff           : 0.13671875
mean abs diff          : 0.06015625
argmax match           : 1.0
Allclose               : False


{'allclose': False,
 'max_abs_diff': 0.13671875,
 'mean_abs_diff': 0.06015624850988388,
 'argmax_match': 1.0}

## 17. 汇总报告：每个函数是否“有帮助”

下面这格会把关键阶段放在一张表里。

你可以根据结果判断：

- **cleanup 有帮助**：Raw -> Clean 后 node count/op pattern 更规整，且 datatype 更完整。
- **materialize 有帮助**：Clean -> Shaped 后 zero-dim 或坏 shape 减少，repaired tensor list 非空。
- **zero-fixed 有帮助**：Shaped -> Fixed 后 zero-dim 变成 0。
- **ConvertQONNXtoFINN 有帮助**：Quant/Trunc 等 QONNX 量化表达变成 FINN 的 MultiThreshold / dataflow-friendly ops。
- **extra streamline 有帮助**：FINN -> Streamlined 后 scalar Add/Mul 和 node count 减少，而且数值验证不坏。

In [19]:

# =========================================================
# 18. Summary table across all stages
# =========================================================

stage_paths = OrderedDict([
    ("raw_qonnx", RAW_QONNX_PATH),
    ("clean_qonnx", CLEAN_QONNX_PATH),
    ("materialized", CLEAN_QONNX_SHAPED_PATH),
    ("zero_fixed", CLEAN_QONNX_SHAPED_FIXED_PATH),
    ("finn", FINN_ONNX_PATH),
    ("streamlined", FINN_ONNX_STREAMLINED_PATH),
])

summary_rows = []

for stage, path in stage_paths.items():
    if not Path(path).is_file():
        continue

    s = graph_stats(path)
    row = {
        "stage": stage,
        "node_count": s["node_count"],
        "zero_dim_tensor_count": s["zero_dim_tensor_count"],
        "scalar_add_mul_count": s["scalar_add_mul_count"],
        "MultiThreshold": s["op_counts"].get("MultiThreshold", 0),
        "Quant": s["op_counts"].get("Quant", 0),
        "Trunc": s["op_counts"].get("Trunc", 0),
        "MatMul": s["op_counts"].get("MatMul", 0),
        "Im2Col": s["op_counts"].get("Im2Col", 0),
        "Pad": s["op_counts"].get("Pad", 0),
        "Transpose": s["op_counts"].get("Transpose", 0),
        "Mul": s["op_counts"].get("Mul", 0),
        "Add": s["op_counts"].get("Add", 0),
    }
    summary_rows.append(row)

if HAS_PANDAS:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
else:
    for row in summary_rows:
        print(row)

print("\nVerification results:")
print("PyTorch vs QONNX fixed:", pt_vs_qonnx_result)
print("QONNX fixed vs FINN streamlined:", qonnx_vs_finn_result)

,stage,node_count,zero_dim_tensor_count,scalar_add_mul_count,MultiThreshold,Quant,Trunc,MatMul,Im2Col,Pad,Transpose,Mul,Add
0,raw_qonnx,94,30,0,0,29,0,0,0,6,6,0,3
1,clean_qonnx,52,0,0,0,29,0,0,0,6,0,0,3
2,materialized,52,0,0,0,29,0,0,0,6,0,0,3
3,zero_fixed,52,0,0,0,29,0,0,0,6,0,0,3
4,finn,76,0,30,13,0,0,0,0,6,0,23,18
5,streamlined,55,0,16,13,0,0,0,0,6,0,16,4



Verification results:
PyTorch vs QONNX fixed: {'allclose': True, 'max_abs_diff': 0.0, 'mean_abs_diff': 0.0, 'argmax_match': 1.0}
QONNX fixed vs FINN streamlined: {'allclose': False, 'max_abs_diff': 0.13671875, 'mean_abs_diff': 0.06015624850988388, 'argmax_match': 1.0}


## 18. 论文里可以提取的证据

你跑完之后，可以把这些结果写进论文或 thesis 的 implementation section：

1. **Static shape materialization**  
   如果 `zero_dim_tensor_count` 下降，或者 repaired tensor list 很多，就说明这一步对 FINN conversion 有直接帮助。

2. **Zero-dimension repair**  
   如果 fixed 后 zero-dim 为 0，就可以写：
   > Remaining zero-valued ONNX dimensions were replaced by symbolic dimensions to avoid invalid static shape propagation.

3. **Scalar/bias absorption**  
   如果 streamlined 后 `scalar_add_mul_count` 或 `Mul/Add` 数量下降，就可以写：
   > Additional streamlining passes were used to absorb scalar scale/bias operations into MultiThreshold nodes.

4. **Functional verification**  
   如果两个 verification 都对齐，就可以写：
   > The graph-normalization flow preserved functional equivalence across PyTorch, QONNX, and FINN-ONNX representations.

注意：如果某个指标没明显改善，就不要强行说它是贡献。可以写成 engineering safeguard 或 compatibility check。